In [ ]:
# Run once per training set:
#   data/train_DAPO-Math-17k.parquet -> icl_corpus/train_DAPO-Math-17k_similar_questions.json
#   data/train_deepcoder.parquet     -> icl_corpus/train_deepcoder_similar_questions.json
TRAIN_DATASET = "data/train_DAPO-Math-17k.parquet"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-8B"
SAVE_FILE = "icl_corpus/train_DAPO-Math-17k_similar_questions.json"
TOPK = 15

In [ ]:
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer

df = pd.read_parquet(TRAIN_DATASET)
prompts = df["prompt"].apply(lambda x: x[0]["content"].strip()).values

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL, model_kwargs={"torch_dtype": torch.bfloat16})

In [ ]:
query_embeddings = model.encode(prompts, 
                                batch_size=16, 
                                prompt_name="query", 
                                device=["cuda:0", "cuda:1", "cuda:2", "cuda:3", "cuda:4", "cuda:5", "cuda:6", "cuda:7"], 
                                show_progress_bar=True
                            )

document_embeddings = model.encode(prompts,
                                   batch_size=16,
                                   device=["cuda:0", "cuda:1", "cuda:2", "cuda:3", "cuda:4", "cuda:5", "cuda:6", "cuda:7"],
                                   show_progress_bar=True
                                )

similarity = model.similarity(query_embeddings, document_embeddings)

In [ ]:
similar_questions = {}
for idx, prompt in enumerate(prompts):
    sim = similarity[idx]
    similar_questions[prompt] = [prompts[k] for k in torch.argsort(sim)[-TOPK:-1].numpy()]

In [ ]:
print("Q1")
print(prompt)
print("Q2")
print(similar_questions[prompt][0])

In [ ]:
import json
with open(SAVE_FILE, 'w') as f:
    json.dump(similar_questions, f)